# 03 · Hero Moment — 语音↔邮件冲突不静默覆盖 (M1)

Email 关键公差 ±0.02mm，Voice 声称可放宽到 ±0.05mm。系统必须 surface 冲突 → HITL，而非任选其一。

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from bootstrap import build_controller, status
from services.config import load_settings, load_policy
print('trunk root:', ROOT)

In [ ]:
from services.intake import extract_rfq, extract_voice_claims, detect_multimodal_conflict
email = 'Please quote 50 pcs 6061 aluminum housings, 100x50x10mm, anodizing. General tolerance IT7, but the critical bore must hold +/-0.02mm.'
voice = '关于那个关键孔径尺寸，其实可以放宽到 ±0.05 毫米，不用做到那么严。'
rfq = extract_rfq(email)
claims = extract_voice_claims(voice)
print('email tolerance_mm=', rfq['tolerance_mm'], 'grade=', rfq['tolerance_grade'])
print('voice claims=', claims)
conf = detect_multimodal_conflict(rfq, claims)
print('conflicts=', json.dumps(conf, ensure_ascii=False))

In [ ]:
ctrl = build_controller()
m1 = json.loads((ROOT/'data'/'golden_scenarios.json').read_text(encoding='utf-8'))['scenarios']
m1 = next(s for s in m1 if s['id']=='M1')
r = ctrl.run(email_text=m1['email'], customer=m1.get('customer'), voice_transcript=m1.get('voice_transcript'))
print('state=', r['state'], '| verify=', r['verification_status'])
print('multimodal_conflicts=', r['multimodal_conflicts'])
print('reasons=', r['reasons'])
print('reply auto_send=', r['reply']['auto_send'], '(仅草稿, 不自动发信)')
assert 'VOICE_EMAIL_CONFLICT' in r['multimodal_conflicts'] and r['verification_status']=='HITL'
print('Hero moment 断言通过 ✅')

In [ ]:
print(r['reply']['subject'])